In [2]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
Groq API Key exists and begins gsk_


In [4]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
llama_url = "http://localhost:8080"

base_client = AsyncOpenAI(base_url=llama_url,api_key="Qwen-35B")
llama_model = OpenAIChatCompletionsModel(model="Qwen3.5-35B-A3B-GGUF",openai_client=base_client)

In [58]:
# First, we need to set the client and then we need to pass the client to OpenAIChatCompletionsModel 
# First, we need to set the client.
#  Then, we need to pass the client to open a chat completions model. 
# That specific API URL will be created. We can then use that model in the agent framework that OpenAI has created for us. 
# This is only for other models. 
# Then, open aand then the model for that specific API URL will be created. 
# We can then use that model in the agent framework that OpenAI has created. This is only for other models than OpenAI.

from openai import base_url


groq_client  = AsyncOpenAI(base_url=GROQ_BASE_URL,api_key=groq_api_key)

groq_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile",openai_client=groq_client)
instruction = "You are a gereal purpose agent"
marketing_agent1 = Agent(name="marketing_a1",instructions=instruction,model=groq_model)
marketing_agent2 = Agent(name="marketing_a1",instructions=instruction,model='gpt-5.2-mini')

 

In [66]:
from agents import guardrail


class NameCheckOutput(BaseModel):
    is_name_in_message:bool
    name:str
    controversial:bool

guardrail_agent = Agent(
    name = "Name Check",
    instructions="""Check whether the user message mentions Donald Trump.

If Trump is mentioned, determine if the question is controversial
Only marks as is controversial as true if it's a personal attack .

Return:
- is_name_in_message
- name
- controversial
""",
    output_type=NameCheckOutput,
    model = "gpt-4o-mini"
)



In [67]:
@input_guardrail
async def guardrail_against_name(ctx,agent,message):
    result = await Runner.run(guardrail_agent,message,context=ctx.context)
    is_name = result.final_output.is_name_in_message
    print("Guardrail output:", result.final_output)

    controversial = result.final_output.controversial
    return GuardrailFunctionOutput(output_info={"Controversy": controversial},tripwire_triggered=controversial) 

In [68]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions="You are a newsletter, so we will be given few questions which you need to answer them based upon the news.",
    model=groq_model,
    input_guardrails=[guardrail_against_name]
    )

message = "Trump achievements"

with trace("Protected"):
    result = await Runner.run(careful_sales_manager, message)

Guardrail output: is_name_in_message=True name='Donald Trump' controversial=False


In [69]:
result.final_output

"**Trump Achievements Newsletter**\n\nIn this edition, we will be highlighting some of the key accomplishments of the 45th President of the United States, Donald Trump. During his presidency, Trump implemented various policies and initiatives that had a significant impact on the country. Here are a few notable achievements:\n\n1. **Economic Growth**: The Trump administration implemented tax cuts, deregulation, and other policies that contributed to a period of economic growth. The unemployment rate fell to historic lows, and the stock market experienced significant gains.\n2. **Judicial Appointments**: Trump appointed two Supreme Court justices, Neil Gorsuch and Brett Kavanaugh, as well as numerous federal judges, which has had a lasting impact on the judiciary.\n3. **Border Security**: The Trump administration implemented various measures to strengthen border security, including the construction of a border wall and increased funding for border patrol agents.\n4. **Veterans' Affairs**

In [5]:
## Assignment

from agents import GuardrailFunctionOutput,input_guardrail,output_guardrail,Agent,trace,Runner
class Checks(BaseModel):
    violence:bool
    sex:bool
    political:bool

input_guard = Agent(
    name = "check_guardrails",
    instructions="""you are a godreils agent. Your work is to check all the categories, like:
- in violence
- sex
- political
- controversial  If it's controversial or having a sentence for violence or sex, then you need to mark them as true""",
    output_type=Checks,
    model = llama_model
)


In [7]:
from openai.types.responses.response_code_interpreter_tool_call_param import Output


class OutChecks(BaseModel):
    isHallucinated:bool
    HateSpeech:bool

outpu_guardrail = Agent(name="output_guard",instructions="You check whether there is hate speech or hallucinated fact in the given text.",output_type=OutChecks,model=llama_model)

@output_guardrail
async def output_function_check(ctx,messages,agent):
    result = await Runner.run(output_guardrail,messages,ctx)
    final_output = result.final_output
    return GuardrailFunctionOutput(output_info={"result":result.final_output},tripwire_triggered=any(final_output.model_dumps().values()))

In [8]:


@input_guardrail
async def checks_for_guardrails(ctx,agent,message):
    print(f"Info context \n {ctx}: \n {ctx.context} \n {agent.name}")
    result = await Runner.run(input_guard,message,context=ctx.context)
    final_result = result.final_output
    print(final_result)
    print(f"Info context \n {ctx}: \n {ctx.context} \n {agent.name}")

    return GuardrailFunctionOutput(output_info={"found_info":result.final_output},tripwire_triggered=any(result.final_output.model_dump().values()))
    
agent_general = Agent(
    name = "gen",
    instructions="your newsletter agent will be giving out the latest information that is out there",
    model = "gpt-4o-mini",
    input_guardrails=[checks_for_guardrails]
)

with trace("assignment_1"):
    result = await Runner.run(agent_general,"Hey Iran is good place to visit")
result.final_output

Info context 
 RunContextWrapper(context=None, usage=Usage(requests=0, input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0)): 
 None 
 gen
violence=False sex=False political=True
Info context 
 RunContextWrapper(context=None, usage=Usage(requests=1, input_tokens=32, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=216, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=248)): 
 None 
 gen


InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire